# Estratégia 1 — Abordagem Heurística (Regras)

## 1. Objetivo

Nesta etapa foi implementado um classificador heurístico baseado em regras manuais, sem a utilização de técnicas de aprendizado de máquina. O objetivo é identificar mensagens fraudulentas (Spam) a partir de características textuais frequentemente associadas a golpes digitais, tais como palavras-gatilho, valores monetários, links suspeitos e padrões de escrita.

## 2. Carregamento do Dataset Pré-processado

O dataset utilizado nesta etapa corresponde ao arquivo pré-processado gerado na etapa anterior do projeto. Esse arquivo contém os textos limpos, lematizados e rotulados nas classes Ham e Spam.

In [ ]:
import pandas as pd
from google.colab import files

print("Selecione o arquivo 'brscams_preprocessado.csv'")
uploaded = files.upload()

nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(nome_arquivo)

print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())

df.head()

Selecione o arquivo 'brscams_preprocessado.csv'


Saving brscams_preprocessado.csv to brscams_preprocessado (1).csv
Dimensões: (450, 10)
Colunas: ['Mensagem', 'Categoria', 'Dataset', 'classe_binaria', 'texto_limpo', 'tokens', 'stems', 'lemas', 'texto_stem', 'texto_lema']


,Mensagem,Categoria,Dataset,classe_binaria,texto_limpo,tokens,stems,lemas,texto_stem,texto_lema
0,"""✨ 3 Sinais Espirituais de que o amor de vocês...",Golpes Baseados Em Relacionamento,Interno,Spam,sinais espirituais de que o amor de voces tem ...,"['sinais', 'espirituais', 'amor', 'voces', 'fu...","['sinal', 'espirit', 'am', 'voc', 'futur', 'vo...","['sinal', 'espiritual', 'amor', 'voce', 'futur...",sinal espirit am voc futur voc sent univers te...,sinal espiritual amor voce futuro voce sentido...
1,"""✅Se você está passando por alguma dificuldade...",Golpes Baseados Em Relacionamento,Interno,Spam,se voce esta passando por alguma dificuldade n...,"['voce', 'passando', 'alguma', 'dificuldade', ...","['voc', 'pass', 'algum', 'dificuldad', 'relaci...","['voce', 'passar', 'algum', 'dificuldade', 're...",voc pass algum dificuldad relacion clic link b...,voce passar algum dificuldade relacionamento c...
2,Sou solteiro e procuro uma namorada para ser m...,Golpes Baseados Em Relacionamento,Interno,Spam,sou solteiro e procuro uma namorada para ser m...,"['solteiro', 'procuro', 'namorada', 'companhei...","['solt', 'procur', 'namor', 'companh', 'sent',...","['solteiro', 'procuro', 'namorar', 'companheir...",solt procur namor companh sent so,solteiro procuro namorar companheira sentir so...
3,"""Eu sou Amanda. 🌹\nMentora espiritual e criado...",Golpes Baseados Em Relacionamento,Interno,Spam,eu sou amanda mentora espiritual e criadora do...,"['amanda', 'mentora', 'espiritual', 'criadora'...","['amand', 'men', 'espirit', 'cri', 'rit', 'sup...","['amando', 'mentora', 'espiritual', 'criadora'...",amand men espirit cri rit supr unia amor ajud ...,amando mentora espiritual criadora ritual Supr...
4,"""Ótimo diia e feliz domingo pra vocês tudos me...",Golpes Baseados Em Relacionamento,Interno,Spam,otimo diia e feliz domingo pra voces tudos meu...,"['otimo', 'diia', 'feliz', 'domingo', 'pra', '...","['otim', 'dii', 'feliz', 'doming', 'pra', 'voc...","['otimo', 'diia', 'feliz', 'domingo', 'pra', '...",otim dii feliz doming pra voc tud am hom arab ...,otimo diia feliz domingo pra voce tur amor hom...


## 3. Definição do Léxico de Palavras-Gatilho

Foi construído um conjunto de palavras frequentemente associadas a golpes, promoções enganosas, falsas oportunidades financeiras, relacionamentos fraudulentos e conteúdos suspeitos observados no dataset.

In [ ]:
palavras_gatilho = [
    'pix',
    'premio',
    'ganhador',
    'ganhou',
    'dinheiro',
    'renda',
    'lucro',
    'investimento',
    'financa',
    'poupanca',
    'cartao',
    'credito',
    'recompensa',
    'gratis',
    'oferta',
    'promocao',
    'bonus',
    'sorteio',
    'teste',
    'amor',
    'namorada',
    'namorado',
    'solteiro',
    'companheira',
    'companheiro',
    'relacionamento',
    'espiritual',
    'mentora',
    'ritual',
    'emagrecimento',
    'trabalho',
    'trabalhar',
    'online',
    'cassino',
    'jogador',
    'hacking',
    'hacker',
    'cura'
]

## 4. Implementação das Regras Heurísticas
Cada mensagem recebe uma pontuação (score) baseada na ocorrência de padrões considerados suspeitos.

As regras implementadas foram:

*   Presença de palavras-gatilho;
*   Presença de URLs;
*   Presença de valores monetários;
*   Uso excessivo de letras maiúsculas.

In [ ]:
import re

def calcular_score(texto_original, texto_lema):

    score = 0

    for palavra in palavras_gatilho:
        if palavra in str(texto_lema):
            score += 1

    if re.search(r'http|www\.', str(texto_original).lower()):
        score += 2

    if re.search(r'R\$|\d+\s*reais|\d+\s*mil',
                 str(texto_original),
                 re.IGNORECASE):
        score += 2

    letras = [c for c in str(texto_original) if c.isalpha()]

    if len(letras) > 0:

        percentual_maiusculas = (
            sum(c.isupper() for c in letras)
            / len(letras)
        )

        if percentual_maiusculas > 0.5:
            score += 1

    return score

## 5. Classificação das Mensagens

A classificação é realizada a partir do score calculado.

Mensagens com score maior ou igual a 2 são classificadas como Spam. Caso contrário, são classificadas como Ham.

In [ ]:
def classificar_heuristica(texto_original, texto_lema):

    score = calcular_score(texto_original, texto_lema)

    if score >= 2:
        return 'Spam'

    return 'Ham'

## 6. Aplicação do Classificador

Nesta etapa o classificador é aplicado a todas as mensagens do dataset.

In [ ]:
df['score'] = df.apply(
    lambda row: calcular_score(
        row['Mensagem'],
        row['texto_lema']
    ),
    axis=1
)

df['predicao_heuristica'] = df.apply(
    lambda row: classificar_heuristica(
        row['Mensagem'],
        row['texto_lema']
    ),
    axis=1
)

df[['Mensagem',
    'classe_binaria',
    'predicao_heuristica',
    'score'
]].head(20)

,Mensagem,classe_binaria,predicao_heuristica,score
0,"""✨ 3 Sinais Espirituais de que o amor de vocês...",Spam,Spam,3
1,"""✅Se você está passando por alguma dificuldade...",Spam,Spam,2
2,Sou solteiro e procuro uma namorada para ser m...,Spam,Spam,3
3,"""Eu sou Amanda. 🌹\nMentora espiritual e criado...",Spam,Spam,5
4,"""Ótimo diia e feliz domingo pra vocês tudos me...",Spam,Ham,1
5,Comentem eu quero 😉👍⬇️⬇️\n #emagrecimento #sau...,Spam,Spam,2
6,💊Um produto perfeito para auxiliar no seu emag...,Spam,Spam,2
7,Faça seu teste grátis por 24 horas! Coloco int...,Spam,Spam,3
8,VOLTO LIMITE DO SEU CARTÃO DE CRÉDITO*\n🟨 ITAU...,Spam,Spam,2
9,Compro Bradesco Net empresa aprovado hoje pago...,Spam,Spam,2


## 7. Distribuição dos Scores
A análise da distribuição dos scores permite compreender como as regras estão sendo aplicadas ao conjunto de dados.

In [ ]:
print(
    df['score']
    .value_counts()
    .sort_index()
)

score
0    257
1     72
2     77
3     34
4      4
5      4
6      2
Name: count, dtype: int64


## 8. Avaliação do Modelo Heurístico

O desempenho foi avaliado utilizando Precisão, Recall e F1-Score.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        df['classe_binaria'],
        df['predicao_heuristica']
    )
)

              precision    recall  f1-score   support

         Ham       0.19      0.81      0.30        75
        Spam       0.88      0.29      0.43       375

    accuracy                           0.37       450
   macro avg       0.53      0.55      0.37       450
weighted avg       0.77      0.37      0.41       450



## 9. Matriz de Confusão

A matriz de confusão permite visualizar os acertos e erros do classificador.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    df['classe_binaria'],
    df['predicao_heuristica']
)

print(cm)

print("\nInterpretação:")
print("- 61 mensagens Ham foram classificadas corretamente.")
print("- 107 mensagens Spam foram classificadas corretamente.")
print("- 14 mensagens Ham foram classificadas incorretamente como Spam.")
print("- 268 mensagens Spam foram classificadas incorretamente como Ham.")

[[ 61  14]
 [268 107]]

Interpretação:
- 61 mensagens Ham foram classificadas corretamente.
- 107 mensagens Spam foram classificadas corretamente.
- 14 mensagens Ham foram classificadas incorretamente como Spam.
- 268 mensagens Spam foram classificadas incorretamente como Ham.


## 10. Exportação das Predições

As predições são exportadas para utilização posterior na etapa de avaliação consolidada do projeto.

In [ ]:
df[
    [
        'Mensagem',
        'classe_binaria',
        'predicao_heuristica',
        'score'
    ]
].to_csv(
    'predicoes_heuristica.csv',
    index=False
)

print("Arquivo salvo com sucesso!")

from google.colab import files

files.download('predicoes_heuristica.csv')

Arquivo salvo com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Considerações

A abordagem heurística apresentou alta interpretabilidade e baixo custo computacional, permitindo identificar explicitamente quais características influenciaram a classificação das mensagens. Entretanto, observou-se dificuldade na detecção de golpes que não continham palavras-gatilho ou padrões previamente definidos, resultando em uma quantidade significativa de falsos negativos. Essa limitação evidencia a necessidade de abordagens baseadas em aprendizado de máquina para capturar padrões linguísticos mais complexos presentes nas mensagens fraudulentas.